# Amazon ML Challenge 2026 — Entity Resolution · Lightning AI Studio · NVIDIA H200 (Jupyter)

**Machine:** Lightning Studio → *Change machine* → **H200** (1x or 8x). Open the **Jupyter** plugin, upload this notebook **and the `er/` folder next to it**, Run All.
Everything (data, checkpoints, outputs) lives in `/teamspace/studios/this_studio`, so it survives a machine switch.

How the H200 is used (all auto-detected in the Config cell):
* GPU: SVD projection (torch, TF32), exact top-K with FAISS sharded over every visible H200 (torch top-K fallback if the
  faiss wheel lacks Hopper kernels), XGBoost `device=cuda` with up to 2 GPUs training the two stage-A models in parallel.
* RAM: the training-row cap scales with machine RAM (up to 80M pairs), bigger processing blocks.
* CPU: every core used for normalisation / text hashing (forkserver workers) and numba feature kernels.

**Cost tip:** stages 1 and 3 are CPU-bound; an H200 does not speed them up. Run stage 1 on a cheap CPU Studio, then
switch to the H200 for stages 2–5 (checkpoints make the switch seamless).

| Stage | FULL, 1x H200 machine (estimate) |
|---|---|
| 0 Download + unzip data | ~5 min |
| 1 Normalise | ~5–12 min (depends on vCPUs) |
| 2 Blocking (forward + reverse + 2-hop search, keys) | ~45–70 min |
| 3 Features (train + test, ~2x more pairs than v4) | ~60–90 min |
| 4 Train + tune | ~20–30 min |
| 5 Predict + write | ~30–45 min |
| 6 Cross-encoder (fine-tune + re-score) | ~30–50 min |

Estimates ±50%; each stage logs its own timings.

Code layout (`er/`): `normalize.py` text views · `text_vectors.py` hashed TF-IDF · `blocking.py` keys + embed + top-K search ·
`stage1_normalise.py` … `stage6_ce.py` one file per stage · `decide.py` / `metric.py` · `run_stage.py` subprocess entry ·
`notebook_utils.py` helpers used by the cells below.

In [ ]:
# ---- Stage 0: environment -----------------------------------------------------------------
import os, sys, json
SRC = os.path.abspath('er')
sys.path.insert(0, SRC)
import notebook_utils as NU
ROOT = NU.setup_env()

In [ ]:
# ---- Data: download / extract student_resource.zip and VERIFY every file -------------------------
S3_URI = None          # or drop student_resource.zip into ROOT yourself; it is used as-is
DATA = NU.prepare_data(ROOT, S3_URI)

In [ ]:
# ---- Config (auto-tuned to the machine) -------------------------------------------------------
FAST = False          # True: 35% of train S1; False: all training entities
SEARCH = 'exact'      # 'exact' (FAISS flat) or 'ivf' (approximate, faster)
RESET = False         # True once = wipe all cached stages (after data problems), then set back to False
# Crash switches (only if a GPU library crashes the stage process on this machine):
os.environ['ER_GPU_PROJECT'] = '1'   # '0' -> SVD projection on CPU instead of torch GPU
os.environ['ER_NO_FAISS'] = '0'      # '1' -> skip FAISS, use torch GPU top-K
cfg = NU.make_cfg(ROOT, DATA, fast=FAST, search=SEARCH, reset=RESET)

In [ ]:
%%time
NU.run_stage(cfg, 'normalise')        # Stage 1 (checkpoint per 400k-row chunk)

In [ ]:
%%time
NU.run_stage(cfg, 'block')            # Stage 2 (checkpoint per view)

In [ ]:
%%time
NU.run_stage(cfg, 'features_train')   # Stage 3a (checkpoint per feature)

In [ ]:
%%time
NU.run_stage(cfg, 'train')            # Stage 4 (checkpoint per model); validation score -> output/validation.json
print(open(f"{cfg['out']}/validation.json").read()[:300])

In [ ]:
# (optional) free disk once models exist: train features are not needed any more
NU.free_train_features(cfg)

In [ ]:
%%time
NU.run_stage(cfg, 'predict')          # Stage 3b + 5: test features, scoring, TSVs in output/

In [ ]:
%%time
# ---- Stage 6: cross-encoder re-ranker (multilingual-e5-small, MIT, 118M params) ------------------
# Fine-tunes on ambiguous TRAIN pairs, re-scores only pairs the GBDT is unsure about, tunes the blend
# weight on the validation entities. Writes output_ce/. If it does not help on validation, w=1 (= output/).
NU.run_stage(cfg, 'ce')
print(open(f"{cfg['out']}/validation_ce.json").read())

In [ ]:
# ---- Pick the better submission by VALIDATION score ------------------------------------------
BEST = NU.pick_best(cfg)

In [ ]:
# ---- Stage 7: validate -------------------------------------------------------------------------
NU.check_submission(ROOT, DATA, BEST)
print(open(f"{cfg['out']}/validation.json").read()[:300])